# RAGulate Tutorial: Inference-Only Regulatory Prediction

This notebook demonstrates how to use RAGulate for post-hoc, literature-grounded gene regulatory inference assesment using a **precomputed document store** and **Mistral (mistralai) LLM**.

## What this notebook does
- Downloads the required RAGulate document store from Zenodo
- Loads a Mistral LLM via Hugging Face
- Runs regulatory edge predictions (TF → target) in a context-specific manner
- Returns a confidence score, LLM justification, and supporting PMIDs

## What this notebook does not do
- No training
- No indexing
- No document preprocessing
- No database construction

This notebook is inference-only.


## 1. Environment requirements

Before running this notebook, ensure:

- Python ≥ 3.9
- Internet access
- Access to a Mistral model on Hugging Face
- A valid `HUGGINGFACE_HUB_TOKEN` set in your environment if required

Example:

```bash
export HUGGINGFACE_HUB_TOKEN=hf_...


In [1]:
from pathlib import Path

## 2. Create workspace and define paths & Download the RAGulate document store 

RAGulate relies on a **precomputed document store** derived from curated regulatory
resources and the biomedical literature.

The file is hosted on **Zenodo** and is required for reproducibility.

- File: `collectri_docs.pkl`
- Source: Zenodo
- Downloaded automatically by this notebook


In [2]:
import requests
from pathlib import Path

DATA_DIR = Path("ragulate_data").resolve()
CACHE_DIR = (DATA_DIR / "pubmed_cache")
DATA_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

# Local paths
SQLITE_PATH = DATA_DIR / "pubmed_cache.sqlite"
NPZ_PATH = CACHE_DIR / "embeddings-sentence-transformers__all-MiniLM-L6-v2.npz"
META_PATH = CACHE_DIR / "embeddings-sentence-transformers__all-MiniLM-L6-v2.meta.json"
HGNC_PATH = DATA_DIR / "hgnc_complete_set.txt"

# Zenodo URLs (minimal set)
SQLITE_URL = "https://zenodo.org/records/18511315/files/pubmed_cache.sqlite?download=1"
NPZ_URL = "https://zenodo.org/records/18511315/files/embeddings-sentence-transformers__all-MiniLM-L6-v2.npz?download=1"
META_URL = "https://zenodo.org/records/18511315/files/embeddings-sentence-transformers__all-MiniLM-L6-v2.meta.json?download=1"
HGNC_URL = "https://zenodo.org/records/18511315/files/hgnc_complete_set.txt?download=1"

def download_if_missing(url: str, path: Path, label: str) -> None:
    if path.exists() and path.stat().st_size > 0:
        print(f"{label} already exists.")
        return

    print(f"Downloading {label} from Zenodo...")
    with requests.get(url, stream=True, timeout=120) as r:
        r.raise_for_status()
        tmp = path.with_suffix(path.suffix + ".tmp")
        with open(tmp, "wb") as f:
            for chunk in r.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    f.write(chunk)
        tmp.replace(path)
    print(f"{label} download complete.")

download_if_missing(SQLITE_URL, SQLITE_PATH, "pubmed_cache.sqlite")
download_if_missing(NPZ_URL, NPZ_PATH, "embeddings .npz")
download_if_missing(META_URL, META_PATH, "embeddings meta.json")
download_if_missing(HGNC_URL, HGNC_PATH, "hgnc_complete_set.txt")


pubmed_cache.sqlite download complete.
embeddings .npz download complete.
embeddings meta.json download complete.
hgnc_complete_set.txt download complete.


## 3. Configure RAGulate to use the downloaded resources

RAGulate uses a centralized configuration file.
Here we explicitly point the pipeline to the downloaded document store to ensure full reproducibility.

In [3]:
import ragulate_bio as ragulate
from ragulate_bio import config as cfg

# 1) Point RAGulate to your downloaded files
cfg.PUBMED_SQLITE = str(SQLITE_PATH)
cfg.PUBMED_CACHE = str(CACHE_DIR)
cfg.HGNC_COMPLETE_SET_FILE = str(HGNC_PATH)

# 2) Ensure embedding cache paths are exactly your downloaded filenames
cfg.EMB_CACHE_NPZ = str(NPZ_PATH)
cfg.EMB_CACHE_META = str(META_PATH)

print("SQLITE:", cfg.PUBMED_SQLITE)
print("CACHE :", cfg.PUBMED_CACHE)
print("NPZ   :", cfg.EMB_CACHE_NPZ)
print("META  :", cfg.EMB_CACHE_META)

# 3) IMPORTANT: reset any already-built singleton/caches (safe to call once)
ragulate.retrieval.ensure_pubmed_retriever(rebuild=True, verbose=True)

SQLITE: /home/mehrdad/research/RAGulate Tutorial/ragulate_data/pubmed_cache.sqlite
CACHE : /home/mehrdad/research/RAGulate Tutorial/ragulate_data/pubmed_cache
NPZ   : /home/mehrdad/research/RAGulate Tutorial/ragulate_data/pubmed_cache/embeddings-sentence-transformers__all-MiniLM-L6-v2.npz
META  : /home/mehrdad/research/RAGulate Tutorial/ragulate_data/pubmed_cache/embeddings-sentence-transformers__all-MiniLM-L6-v2.meta.json
[rebuild] invalidating encoder corpora and global state
[init] Sentence embedder -> sentence-transformers/all-MiniLM-L6-v2
[ready] Sentence embedder initialized
[global-corpus] loaded cache: 38021 docs, dim=384
[retriever] create new instance (encoder=minilm, topk=8)


(None, <ragulate_bio.retrieval.PubMedRetriever at 0x7f0349f87b80>)

## 4. Load the Mistral language model

RAGulate supports large language models for final regulatory assessment.

In this tutorial, we use a **Mistral (mistralai)** model via Hugging Face.

Default model:
- `mistralai/Mistral-7B-Instruct-v0.2`

In [4]:
tokenizer, model = ragulate.llm_models.get_mistral()

[init] Mistral -> mistralai/Mistral-7B-Instruct-v0.2


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the cpu.


[ready] Mistral initialized


## 5. Optional sanity check

This step verifies that the Mistral model is correctly loaded and responding.


In [5]:
print(
    ragulate.llm_models._llm_generate(
        tokenizer,
        model,
        "Answer with yes or no: Does TP53 regulate CDKN1A?",
        max_new_tokens=32,
    )
)

Answer with yes or no: Does TP53 regulate CDKN1A?

Yes, TP53 regulates CDKN1A. TP53 is a transcription factor that can bind to the CDK


## 6. Define regulatory queries

Each row corresponds to a putative regulatory edge defined by:
- Transcription factor
- Target gene
- Biological context (cell type, tissue, or condition)


In [6]:
import pandas as pd

edges = pd.DataFrame([
    {"tf": "PAX5",  "target": "CD19",  "context": "naive B cell"},
    {"tf": "TBX21", "target": "GZMB",  "context": "NK cell"},
    {"tf": "SPI1",  "target": "CSF1R", "context": "monocyte"},
])

## 7. Run RAGulate inference

RAGulate combines:
- Hybrid retrieval (BM25 + embedding similarity)
- Alias-aware querying
- LLM-based regulatory assessment
- Literature-backed evidence aggregation


In [7]:
import ragulate_bio.retrieval as R

R._BM25_INDEX = None
R._BM25_TOKENS = None
R._BM25_PMIDS = None
if hasattr(R, "_BM25_SOURCE_PATH"):
    R._BM25_SOURCE_PATH = None


In [8]:
results = ragulate.pipeline.run_ragulate_inference(
    edges_df=edges,
    method_type="hybrid",
    encoder_name="minilm",
    top_k_docs=20,
    use_llm=True,
    llm_name=ragulate.config.MISTRAL_MODEL_NAME,
    classify=True,
    use_aliases=True,
)

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


## 8. Inspect results


In [9]:
results[
    [
        "tf",
        "target",
        "context",
        "score",
        "llm_summary",
        "retrieved_pmids",
    ]
]


,tf,target,context,score,llm_summary,retrieved_pmids
0,PAX5,CD19,naive B cell,0.865013,"PAX5, the transcription factor that encodes th...","[8639790, 9722295, 9545244, 7532151, 12052884,..."
1,TBX21,GZMB,NK cell,0.864883,"Transcription factor TBX21, also known as T-be...","[25352127, 8219227, 24752800, 15991363, 212720..."
2,SPI1,CSF1R,monocyte,0.359616,The transcription factor SPI1 (also known as P...,"[17116688, 8302571, 7964503, 8981363, 9506963,..."


## 9. Save predictions


To save in tabular `.csv` format:

In [10]:
results.to_csv("ragulate_predictions.csv", index=False)

To save in web `.html` format:

In [11]:
results.to_html("ragulate_predictions.html", index=False)

## Summary

You have now:
- Downloaded all required RAGulate resources automatically
- Loaded your own Mistral LLM
- Queried regulatory hypotheses
- Obtained literature-grounded predictions with confidence scores
